In [26]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SupplyChainAnalytics") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [27]:
df_drivers = spark.read.csv("drivers.csv", header=True, inferSchema=True)
df_trips = spark.read.csv("trips.csv", header=True, inferSchema=True)
df_loads = spark.read.csv("loads.csv", header=True, inferSchema=True)
df_routes = spark.read.csv("routes.csv", header=True, inferSchema=True)
df_customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
df_fuel = spark.read.csv("fuel_purchases.csv", header=True, inferSchema=True)
df_maintenance = spark.read.csv("maintenance_records.csv", header=True, inferSchema=True)
df_delivery = spark.read.csv("delivery_events.csv", header=True, inferSchema=True)
df_safety = spark.read.csv("safety_incidents.csv", header=True, inferSchema=True)

In [3]:
df_trips.show()
df_customers.show()
df_loads.show()

+------------+------------+---------+--------+----------+-------------+---------------------+---------------------+-----------------+-----------+---------------+-----------+
|     trip_id|     load_id|driver_id|truck_id|trailer_id|dispatch_date|actual_distance_miles|actual_duration_hours|fuel_gallons_used|average_mpg|idle_time_hours|trip_status|
+------------+------------+---------+--------+----------+-------------+---------------------+---------------------+-----------------+-----------+---------------+-----------+
|TRIP00000001|LOAD00000001| DRV00117|TRK00035|  TRL00167|   2022-01-01|                 1314|                 26.2|            183.8|       7.15|            3.5|  Completed|
|TRIP00000002|LOAD00000002| DRV00141|TRK00108|  TRL00082|   2022-01-01|                  515|                  8.6|             93.6|        5.5|            8.3|  Completed|
|TRIP00000003|LOAD00000003| DRV00032|TRK00031|  TRL00138|   2022-01-01|                 2509|                 45.0|            339

In [28]:
from pyspark.sql.functions import col
df_trips=df_trips.dropna(subset= "trip_id")
df_trips=df_trips.dropDuplicates()


In [29]:
from pyspark.sql.functions import col
df_loads= df_loads.withColumn("revenue" , col("revenue").cast("double"))

In [15]:
df_loads.show()

+------------+-----------+--------+----------+------------+----------+------+-------+--------------+-------------------+-----------+------------+
|     load_id|customer_id|route_id| load_date|   load_type|weight_lbs|pieces|revenue|fuel_surcharge|accessorial_charges|load_status|booking_type|
+------------+-----------+--------+----------+------------+----------+------+-------+--------------+-------------------+-----------+------------+
|LOAD00000001|  CUST00183|RTE00019|2022-01-01|     Dry Van|     19178|    13|3045.23|        406.72|                100|  Completed|        Spot|
|LOAD00000002|  CUST00076|RTE00058|2022-01-01|     Dry Van|     27761|    22|1224.48|         98.61|                  0|  Completed|   Dedicated|
|LOAD00000003|  CUST00027|RTE00048|2022-01-01|Refrigerated|     35594|    16|7171.12|        792.88|                  0|  Completed|        Spot|
|LOAD00000004|  CUST00088|RTE00013|2022-01-01|Refrigerated|     33274|    10| 1308.2|        141.33|                 50|  Co

In [30]:
df_master= df_trips.join(df_loads, "load_id", "inner") \
.join(df_drivers, "driver_id", "left").join(df_routes, "route_id", "left").join(df_customers, "customer_id", "left")
                   


In [87]:
display(df_master)

DataFrame[customer_id: string, route_id: string, driver_id: string, load_id: string, trip_id: string, truck_id: string, trailer_id: string, dispatch_date: date, actual_distance_miles: int, actual_duration_hours: double, fuel_gallons_used: double, average_mpg: double, idle_time_hours: double, trip_status: string, load_date: date, load_type: string, weight_lbs: int, pieces: int, revenue: double, fuel_surcharge: double, accessorial_charges: int, load_status: string, booking_type: string, first_name: string, last_name: string, hire_date: date, termination_date: date, license_number: string, license_state: string, date_of_birth: date, home_terminal: string, employment_status: string, cdl_class: string, years_experience: int, origin_city: string, origin_state: string, destination_city: string, destination_state: string, typical_distance_miles: int, base_rate_per_mile: double, fuel_surcharge_rate: double, typical_transit_days: int, customer_name: string, customer_type: string, credit_terms_da

In [88]:
df_master.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- trailer_id: string (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- actual_distance_miles: integer (nullable = true)
 |-- actual_duration_hours: double (nullable = true)
 |-- fuel_gallons_used: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- idle_time_hours: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable =

In [38]:
df_master.select("customer_id", "driver_id", "revenue", "trip_status").show(5)

+-----------+---------+-------+-----------+
|customer_id|driver_id|revenue|trip_status|
+-----------+---------+-------+-----------+
|  CUST00049| DRV00065| 5914.9|  Completed|
|  CUST00075| DRV00129|4759.42|  Completed|
|  CUST00127| DRV00132|1206.59|  Completed|
|  CUST00089| DRV00071|1286.22|  Completed|
|  CUST00022| DRV00094|3521.84|  Completed|
+-----------+---------+-------+-----------+
only showing top 5 rows


In [31]:
from pyspark.sql.functions import sum, avg
driver_performan= df_master.groupBy("driver_id").agg(sum("revenue").alias("Toatal_Rev"), avg("actual_distance_miles").alias("avg_distance"))

# Driver Performance

In [40]:
driver_performan.show()

+---------+------------------+------------------+
|driver_id|        Toatal_Rev|      avg_distance|
+---------+------------------+------------------+
| DRV00125|2094600.8599999994|1416.7343976777938|
| DRV00124|2084929.8399999996|1382.9615384615386|
| DRV00066|2180636.1999999997|1366.7503392130259|
| DRV00141|2096526.4500000002|1455.1649175412294|
| DRV00105|2106838.8799999994|1434.1251840942562|
| DRV00143|2098126.2199999997|1460.6571856287426|
| DRV00075|1989674.7000000002|1426.9753466872112|
| DRV00024|        2166574.42|1426.2612994350281|
| DRV00003|2146977.4699999997|1492.2230428360413|
| DRV00081|2027839.2900000003| 1421.490990990991|
| DRV00090|2111290.9300000006|1421.8731988472623|
| DRV00009|        2073798.94|1444.7174887892377|
| DRV00011| 2153396.549999999| 1434.058908045977|
| DRV00016|        2278574.88|1464.3693820224719|
| DRV00010|        2144658.93|1407.7799717912553|
| DRV00074|1898861.4200000002|1394.4636075949368|
| DRV00110|2003902.8699999996|1445.7079510703363|


# Route Profitability"

In [32]:
from pyspark.sql.functions import sum, avg
route_profit= df_master.groupBy("route_id").agg(sum("revenue").alias("Toatal_Rev"), avg("actual_distance_miles").alias("avg_distance"))

In [20]:
route_profit.show()

+--------+------------------+------------------+
|route_id|        Toatal_Rev|      avg_distance|
+--------+------------------+------------------+
|RTE00006|2105438.9899999993| 694.3383561643835|
|RTE00034| 7264971.310000001|2400.6528532608695|
|RTE00056| 6234118.289999999|1769.7511896668932|
|RTE00043|3616131.1599999983|1604.8433333333332|
|RTE00048| 9586536.899999999|2400.8059016393445|
|RTE00002| 2154291.120000001| 718.5476029709656|
|RTE00035|2990648.7499999977|1263.9014573213046|
|RTE00055|         5170899.4|1469.3312543312543|
|RTE00053| 2529485.879999999| 800.3838383838383|
|RTE00008|2915299.8600000013| 784.9486652977413|
|RTE00037| 5770647.800000001| 2637.480027081923|
|RTE00018|2023437.2099999988| 689.2891483516484|
|RTE00014| 3840447.090000001|1241.2353732347008|
|RTE00058|1682085.0099999988| 534.2374301675977|
|RTE00020| 5712509.239999999|2237.6700336700337|
|RTE00015|393428.95000000007| 94.26284970722186|
|RTE00054| 4603410.160000002| 1603.969429347826|
|RTE00042| 9070885.0

# Fuel efficiency

In [41]:
df_fuel.show(5)

+----------------+------------+--------+---------+-------------------+-------------+--------------+-------+----------------+----------+----------------+
|fuel_purchase_id|     trip_id|truck_id|driver_id|      purchase_date|location_city|location_state|gallons|price_per_gallon|total_cost|fuel_card_number|
+----------------+------------+--------+---------+-------------------+-------------+--------------+-------+----------------+----------+----------------+
|    FUEL00000001|TRIP00051284|TRK00045| DRV00102|2023-10-22 05:00:00|     Columbus|            MN|  131.6|           3.399|    447.31|        FC567161|
|    FUEL00000002|TRIP00073723|TRK00013|     NULL|2024-08-04 08:00:00|     New York|            AZ|  139.9|            3.18|    444.88|        FC717910|
|    FUEL00000003|TRIP00018286|TRK00024| DRV00142|2022-08-23 13:00:00|      Seattle|            NE|  189.3|           3.804|     720.1|        FC912816|
|    FUEL00000004|TRIP00008465|TRK00018| DRV00047|2022-04-20 00:00:00|      Memphi

In [58]:
from pyspark.sql.functions import sum, round

fuel_eff = df_fuel.groupBy("trip_id") \
    .agg(round(sum("gallons"), 2).alias("total_fuel_used"))

fuel_eff.show()

+------------+---------------+
|     trip_id|total_fuel_used|
+------------+---------------+
|TRIP00073723|          444.4|
|TRIP00052921|          276.4|
|TRIP00056824|          602.3|
|TRIP00014898|          307.2|
|TRIP00070753|          483.3|
|TRIP00048089|          531.4|
|TRIP00073196|          228.1|
|TRIP00024611|          411.1|
|TRIP00077065|          719.3|
|TRIP00072728|          916.2|
|TRIP00030642|          159.0|
|TRIP00036577|          773.6|
|TRIP00047095|           62.6|
|TRIP00064891|          870.5|
|TRIP00006150|          949.3|
|TRIP00070166|          377.6|
|TRIP00043939|          748.1|
|TRIP00067778|          712.9|
|TRIP00039918|          167.3|
|TRIP00004426|          113.9|
+------------+---------------+
only showing top 20 rows


In [59]:
df_efficiency= df_master.join(fuel_eff, "trip_id","left").withColumn("mpg", round(col("actual_distance_miles")/col("total_fuel_used"),2))

In [60]:
df_efficiency.select("trip_id", "mpg").show()

+------------+-----+
|     trip_id|  mpg|
+------------+-----+
|TRIP00000008|22.14|
|TRIP00000009| NULL|
|TRIP00000011| 4.98|
|TRIP00000015| 8.54|
|TRIP00000017| 4.73|
|TRIP00000018| 2.51|
|TRIP00000019| 8.42|
|TRIP00000022| 3.88|
|TRIP00000026| 1.37|
|TRIP00000031| 5.96|
|TRIP00000036| NULL|
|TRIP00000039|11.86|
|TRIP00000045| 7.85|
|TRIP00000063| 1.54|
|TRIP00000069|12.01|
|TRIP00000075|12.46|
|TRIP00000087| 2.49|
|TRIP00000120| 2.12|
|TRIP00000126| NULL|
|TRIP00000131| NULL|
+------------+-----+
only showing top 20 rows


Customer Revenue

In [33]:
customer_rev = df_master.groupBy("customer_id") \
    .agg(sum("revenue").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

In [62]:
customer_rev.show()

+-----------+------------------+
|customer_id|     total_revenue|
+-----------+------------------+
|  CUST00200|1544419.8100000005|
|  CUST00181|1542321.0199999998|
|  CUST00077|1521982.0700000003|
|  CUST00097|1487129.3100000003|
|  CUST00122|1483188.8999999997|
|  CUST00028|        1481527.84|
|  CUST00110|1479584.7300000002|
|  CUST00101|1477854.3199999998|
|  CUST00124|1472131.3100000003|
|  CUST00196|1471132.9000000006|
|  CUST00164|1470009.7099999997|
|  CUST00035|1461214.0300000005|
|  CUST00133|1455346.0199999998|
|  CUST00160|1442083.3099999996|
|  CUST00136|        1438452.71|
|  CUST00135|1437776.9299999997|
|  CUST00076|1437038.5099999998|
|  CUST00062|        1431495.57|
|  CUST00134|1430000.0599999996|
|  CUST00111|1427110.3699999999|
+-----------+------------------+
only showing top 20 rows


# Customer Revenue

In [34]:
from pyspark.sql.functions import desc
customer_rev=df_master.groupBy("customer_id")\
     .agg(sum(col("revenue")).alias("Total_rev"))\
     .orderBy(desc("Total_rev"))

In [73]:
customer_rev.show()

+-----------+------------------+
|customer_id|         Total_rev|
+-----------+------------------+
|  CUST00200|1544419.8100000005|
|  CUST00181|1542321.0199999998|
|  CUST00077|1521982.0700000003|
|  CUST00097|1487129.3100000003|
|  CUST00122|1483188.8999999997|
|  CUST00028|        1481527.84|
|  CUST00110|1479584.7300000002|
|  CUST00101|1477854.3199999998|
|  CUST00124|1472131.3100000003|
|  CUST00196|1471132.9000000006|
|  CUST00164|1470009.7099999997|
|  CUST00035|1461214.0300000005|
|  CUST00133|1455346.0199999998|
|  CUST00160|1442083.3099999996|
|  CUST00136|        1438452.71|
|  CUST00135|1437776.9299999997|
|  CUST00076|1437038.5099999998|
|  CUST00062|        1431495.57|
|  CUST00134|1430000.0599999996|
|  CUST00111|1427110.3699999999|
+-----------+------------------+
only showing top 20 rows


In [35]:
#alternative
customer_rev=df_master.groupBy("customer_id")\
     .agg(sum("revenue").alias("Total_rev"))\
     .orderBy (col("Total_rev").desc())

In [36]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
window_spec= Window.partitionBy ("route_id").orderBy(col("revenue").desc())
df_ranked= df_master.withColumn("Rank",rank().over(window_spec))

# Revenue wise Rank of routes

In [81]:
df_ranked.select("route_id","revenue", "Rank").show()

+--------+-------+----+
|route_id|revenue|Rank|
+--------+-------+----+
|RTE00001|1323.24|   1|
|RTE00001|1323.16|   2|
|RTE00001|1322.77|   3|
|RTE00001| 1322.5|   4|
|RTE00001|1322.44|   5|
|RTE00001|1322.38|   6|
|RTE00001|1322.32|   7|
|RTE00001|1322.13|   8|
|RTE00001|1322.12|   9|
|RTE00001|1321.92|  10|
|RTE00001|1321.92|  10|
|RTE00001|1320.79|  12|
|RTE00001|1320.76|  13|
|RTE00001|1320.76|  13|
|RTE00001| 1320.7|  15|
|RTE00001|1320.59|  16|
|RTE00001|1320.46|  17|
|RTE00001|1320.42|  18|
|RTE00001|1320.31|  19|
|RTE00001|1320.29|  20|
+--------+-------+----+
only showing top 20 rows


In [82]:
df_master.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- trailer_id: string (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- actual_distance_miles: integer (nullable = true)
 |-- actual_duration_hours: double (nullable = true)
 |-- fuel_gallons_used: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- idle_time_hours: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable =

In [85]:
df_master.select("customer_id","route_id","contract_start_date").show()

+-----------+--------+-------------------+
|customer_id|route_id|contract_start_date|
+-----------+--------+-------------------+
|  CUST00049|RTE00016|         2020-01-30|
|  CUST00075|RTE00025|         2021-04-20|
|  CUST00127|RTE00058|         2021-03-18|
|  CUST00089|RTE00006|         2021-01-10|
|  CUST00022|RTE00054|         2021-05-23|
|  CUST00093|RTE00038|         2020-04-28|
|  CUST00092|RTE00012|         2021-03-26|
|  CUST00018|RTE00009|         2021-06-12|
|  CUST00033|RTE00022|         2020-11-22|
|  CUST00135|RTE00042|         2020-02-27|
|  CUST00081|RTE00020|         2021-03-06|
|  CUST00044|RTE00025|         2021-10-02|
|  CUST00060|RTE00031|         2021-09-14|
|  CUST00170|RTE00008|         2020-12-28|
|  CUST00095|RTE00026|         2021-07-21|
|  CUST00070|RTE00028|         2021-12-11|
|  CUST00078|RTE00022|         2020-01-09|
|  CUST00085|RTE00002|         2021-06-21|
|  CUST00094|RTE00035|         2020-06-13|
|  CUST00143|RTE00006|         2020-08-22|
+----------

# Monthly trend of revenue

In [37]:
from pyspark.sql.functions import month

df_time = df_master.withColumn("month", month("contract_start_date"))

monthly_trend = df_time.groupBy("month") \
    .agg(sum("revenue").alias("monthly_revenue"))

In [93]:
df_time.select("month" , "revenue").show()

+-----+-------+
|month|revenue|
+-----+-------+
|    1| 5914.9|
|    4|4759.42|
|    3|1206.59|
|    1|1286.22|
|    5|3521.84|
|    4|3622.99|
|    3|6009.91|
|    6|1754.79|
|   11|1091.98|
|    2|5386.12|
|    3|4255.48|
|   10|5362.03|
|    9| 1397.3|
|   12|2294.59|
|    7|2969.62|
|   12|4476.08|
|    1|1212.18|
|    6|1417.56|
|    6|2259.52|
|    8|1361.45|
+-----+-------+
only showing top 20 rows


In [96]:
df_safety.select("incident_id", "at_fault_flag").show()


+-----------+-------------+
|incident_id|at_fault_flag|
+-----------+-------------+
|INC00000001|         true|
|INC00000002|        false|
|INC00000003|         true|
|INC00000004|         true|
|INC00000005|        false|
|INC00000006|        false|
|INC00000007|         true|
|INC00000008|        false|
|INC00000009|        false|
|INC00000010|        false|
|INC00000011|         true|
|INC00000012|        false|
|INC00000013|        false|
|INC00000014|        false|
|INC00000015|        false|
|INC00000016|         true|
|INC00000017|        false|
|INC00000018|        false|
|INC00000019|        false|
|INC00000020|        false|
+-----------+-------------+
only showing top 20 rows


# Safety analysis

In [97]:
df_safety.printSchema()

root
 |-- incident_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- incident_date: timestamp (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- at_fault_flag: boolean (nullable = true)
 |-- injury_flag: boolean (nullable = true)
 |-- vehicle_damage_cost: double (nullable = true)
 |-- cargo_damage_cost: double (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- preventable_flag: boolean (nullable = true)
 |-- description: string (nullable = true)



In [38]:
from pyspark.sql.functions import count, when

safety_metrics = df_safety.groupBy("driver_id").agg(
    sum("cargo_damage_cost").alias("total_incident_cost"),
    count(when(col("at_fault_flag") == True, 1)).alias("at_fault_cases")
)

In [102]:
safety_metrics.show()

+---------+-------------------+--------------+
|driver_id|total_incident_cost|at_fault_cases|
+---------+-------------------+--------------+
| DRV00003|            6604.51|             1|
| DRV00066|                0.0|             1|
| DRV00088|                0.0|             2|
| DRV00011|             8080.5|             0|
| DRV00054|                0.0|             1|
| DRV00090|                0.0|             1|
| DRV00124|                0.0|             2|
| DRV00141|                0.0|             0|
| DRV00105|                0.0|             0|
| DRV00016|                0.0|             0|
| DRV00143|           37531.43|             1|
| DRV00074|                0.0|             0|
| DRV00081|           28390.39|             1|
| DRV00005|                0.0|             0|
| DRV00079|                0.0|             1|
| DRV00045|                0.0|             0|
| DRV00018|             408.98|             0|
| DRV00094| 60483.649999999994|             1|
| DRV00029|  

In [39]:
#alternative
from pyspark.sql.functions import sum, col

safety_metrics = df_safety.groupBy("driver_id").agg(
    sum("cargo_damage_cost").alias("total_incident_cost"),
    sum(col("at_fault_flag").cast("int")).alias("at_fault_cases")
)

In [110]:
safety_metrics.orderBy(col("total_incident_cost").desc()).show(5)

+---------+-------------------+--------------+
|driver_id|total_incident_cost|at_fault_cases|
+---------+-------------------+--------------+
| DRV00142|            88444.2|             2|
| DRV00012|           84932.55|             0|
| DRV00094| 60483.649999999994|             1|
| DRV00013|           54804.94|             0|
| DRV00098|           49744.07|             0|
+---------+-------------------+--------------+
only showing top 5 rows


# Maintenance Cost per Truck

In [10]:
df_maintenance.printSchema()

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: date (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- odometer_reading: integer (nullable = true)
 |-- labor_hours: double (nullable = true)
 |-- labor_cost: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- facility_location: string (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- service_description: string (nullable = true)



In [40]:

maintennace_cost= df_maintenance.groupBy("truck_id").agg(sum("labor_cost" )
                                                              , sum("parts_cost"))

In [20]:
maintennace_cost.show()

+--------+------------------+------------------+
|truck_id|   sum(labor_cost)|   sum(parts_cost)|
+--------+------------------+------------------+
|TRK00017| 8625.810000000001|          37640.03|
|TRK00076|           7267.25|          22865.96|
|TRK00052|11163.009999999998| 52594.20000000001|
|TRK00044|          18003.28|          51821.95|
|TRK00070|           8192.87|          27949.45|
|TRK00117|           4804.04|20774.989999999998|
|TRK00025|10422.859999999997|          33659.26|
|TRK00027| 9760.190000000002|          34470.37|
|TRK00098|           7643.88| 33675.06999999999|
|TRK00084| 8787.770000000002|25956.020000000004|
|TRK00064| 9158.469999999998|          37137.89|
|TRK00061|           9810.54| 34183.03999999999|
|TRK00080| 6978.390000000001|          37916.23|
|TRK00103|12559.450000000003|          42426.77|
|TRK00049|            8562.6|24682.829999999994|
|TRK00028|           7800.55|36091.799999999996|
|TRK00016| 5685.759999999999|          30762.29|
|TRK00011|10209.3000

In [41]:
totl_main_cost= maintennace_cost.withColumn("Total_cost",col("sum(labor_cost)")+col("sum(parts_cost)"))

In [42]:
totl_main_cost.orderBy(col("Total_cost").desc()).show()

+--------+------------------+------------------+-----------------+
|truck_id|   sum(labor_cost)|   sum(parts_cost)|       Total_cost|
+--------+------------------+------------------+-----------------+
|TRK00003| 18587.94000000001|          71573.48|90161.42000000001|
|TRK00073|16113.529999999999|60460.459999999985|76573.98999999999|
|TRK00081|14856.989999999996|59035.729999999996|73892.71999999999|
|TRK00099|15214.049999999997|          58017.72|         73231.77|
|TRK00014|18455.050000000003| 53346.77000000001|         71801.82|
|TRK00075|15547.349999999999| 55526.28999999999|71073.63999999998|
|TRK00118|          15815.53|55190.240000000005|         71005.77|
|TRK00026|14358.059999999996|56350.420000000006|         70708.48|
|TRK00044|          18003.28|          51821.95|         69825.23|
|TRK00038|12719.480000000001| 56269.65999999998|68989.13999999998|
|TRK00032|          12591.56| 56170.77999999999|         68762.34|
|TRK00040|13122.259999999997| 54435.47999999999|67557.73999999

# Performance Optimization (repartition)

In [43]:
df_master=df_master.repartition(4, "route_id")

In [61]:
df_master.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- trailer_id: string (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- actual_distance_miles: integer (nullable = true)
 |-- actual_duration_hours: double (nullable = true)
 |-- fuel_gallons_used: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- idle_time_hours: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable =

# top 3 drivers per route

In [52]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window
window_spec= Window.partitionBy("driver_id").orderBy(col("revenue").desc())
driver_rank= df_master.withColumn("rank", dense_rank().over (window_spec)).filter(col("rank")<=3)

In [54]:
driver_rank.select("driver_id", "revenue", "rank").show()

+---------+-------+----+
|driver_id|revenue|rank|
+---------+-------+----+
|     NULL| 8025.0|   1|
|     NULL|7964.13|   2|
|     NULL|7915.76|   3|
| DRV00001|7841.71|   1|
| DRV00001|7599.74|   2|
| DRV00001|7554.82|   3|
| DRV00002|7922.61|   1|
| DRV00002|7888.65|   2|
| DRV00002|7825.61|   3|
| DRV00003|7985.22|   1|
| DRV00003|7568.07|   2|
| DRV00003|7495.72|   3|
| DRV00004|7898.91|   1|
| DRV00004|7859.47|   2|
| DRV00004|7706.36|   3|
| DRV00005|7882.46|   1|
| DRV00005|7715.07|   2|
| DRV00005|7608.14|   3|
| DRV00006|8105.31|   1|
| DRV00006|8042.97|   2|
+---------+-------+----+
only showing top 20 rows


# Detect delayed deliveries

In [56]:
df_delivery.show(5)

+-----------+------------+------------+----------+-----------+--------------------+--------------------+-----------------+------------+-------------+--------------+
|   event_id|     load_id|     trip_id|event_type|facility_id|  scheduled_datetime|     actual_datetime|detention_minutes|on_time_flag|location_city|location_state|
+-----------+------------+------------+----------+-----------+--------------------+--------------------+-----------------+------------+-------------+--------------+
|EVT00000001|LOAD00000001|TRIP00000001|    Pickup|   FAC00034| 2022-01-01 18:00:00|2022-01-01 20:58:...|                0|       false|      Houston|            TX|
|EVT00000002|LOAD00000001|TRIP00000001|  Delivery|   FAC00046|2022-01-02 23:10:...|2022-01-02 21:30:...|              230|        true|      Detroit|            MI|
|EVT00000003|LOAD00000002|TRIP00000002|    Pickup|   FAC00015| 2022-01-01 18:00:00|2022-01-01 17:37:...|               62|        true|  Kansas City|            MO|
|EVT000000

In [57]:
delay= df_delivery.filter(col("actual_datetime")>col("scheduled_datetime"))

In [58]:
delay.show(5)

+-----------+------------+------------+----------+-----------+--------------------+--------------------+-----------------+------------+-------------+--------------+
|   event_id|     load_id|     trip_id|event_type|facility_id|  scheduled_datetime|     actual_datetime|detention_minutes|on_time_flag|location_city|location_state|
+-----------+------------+------------+----------+-----------+--------------------+--------------------+-----------------+------------+-------------+--------------+
|EVT00000001|LOAD00000001|TRIP00000001|    Pickup|   FAC00034| 2022-01-01 18:00:00|2022-01-01 20:58:...|                0|       false|      Houston|            TX|
|EVT00000004|LOAD00000002|TRIP00000002|  Delivery|   FAC00050|2022-01-02 02:13:...|2022-01-02 05:55:...|               67|       false| Indianapolis|            IN|
|EVT00000007|LOAD00000004|TRIP00000004|    Pickup|   FAC00029| 2022-01-01 17:00:00|2022-01-01 19:32:...|               50|       false|      Phoenix|            AZ|
|EVT000000

# Running Total

In [73]:
from pyspark.sql.functions import sum
from pyspark.sql.functions import round
from pyspark.sql.window import Window
window_spec= Window.partitionBy("driver_id").orderBy("load_date")
df_running = df_master.withColumn(
    "running_revenue",
    round(sum("revenue").over(window_spec),2)
)

In [84]:

from pyspark.sql.functions import col

df_running_filtered = df_running \
    .filter(col("driver_id").isNotNull()) \
    .orderBy(col("running_revenue").desc())

df_running_filtered.show(5)

+-----------+--------+---------+------------+------------+--------+----------+-------------+---------------------+---------------------+-----------------+-----------+---------------+-----------+----------+------------+----------+------+-------+--------------+-------------------+-----------+------------+----------+---------+----------+----------------+--------------+-------------+-------------+-------------+-----------------+---------+----------------+-----------+------------+----------------+-----------------+----------------------+------------------+-------------------+--------------------+--------------------+-------------+-----------------+--------------------+--------------+-------------------+------------------------+---------------+
|customer_id|route_id|driver_id|     load_id|     trip_id|truck_id|trailer_id|dispatch_date|actual_distance_miles|actual_duration_hours|fuel_gallons_used|average_mpg|idle_time_hours|trip_status| load_date|   load_type|weight_lbs|pieces|revenue|fuel_su

# Compare current vs previous trip

In [75]:
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("driver_id").orderBy("load_date")

df_lag= df_master.withColumn(
    "prev_revenue",lag("revenue",1).over(window_spec)
)

In [81]:
df_lag.filter(col("driver_id").isNotNull()) \
      .select("driver_id", "load_date", "revenue", "prev_revenue") \
      .show()

+---------+----------+-------+------------+
|driver_id| load_date|revenue|prev_revenue|
+---------+----------+-------+------------+
| DRV00001|2022-01-01|3290.19|        NULL|
| DRV00001|2022-01-03|5365.21|     3290.19|
| DRV00001|2022-01-04|2858.53|     5365.21|
| DRV00001|2022-01-05|1122.53|     2858.53|
| DRV00001|2022-01-06| 888.01|     1122.53|
| DRV00001|2022-01-06|6856.94|      888.01|
| DRV00001|2022-01-07|3590.65|     6856.94|
| DRV00001|2022-01-08| 145.15|     3590.65|
| DRV00001|2022-01-08| 6265.4|      145.15|
| DRV00001|2022-01-12|3784.22|      6265.4|
| DRV00001|2022-01-12|2351.74|     3784.22|
| DRV00001|2022-01-14|2974.78|     2351.74|
| DRV00001|2022-01-15|1226.19|     2974.78|
| DRV00001|2022-01-15|1265.82|     1226.19|
| DRV00001|2022-01-17|3733.96|     1265.82|
| DRV00001|2022-01-17|3141.55|     3733.96|
| DRV00001|2022-01-17|4242.97|     3141.55|
| DRV00001|2022-01-18| 826.68|     4242.97|
| DRV00001|2022-01-18|1327.43|      826.68|
| DRV00001|2022-01-20|3152.26|  

# Salting Technique

In [85]:
from pyspark.sql.functions import rand, concat

df_skewed = df_master.withColumn("salt", (rand()*5).cast("int"))

df_new = df_skewed.withColumn(
    "route_salt",
    concat(col("route_id"), col("salt"))
)

# High revenue trips

In [90]:
from pyspark.sql.functions import when
df_flag= df_master.withColumn("high_value", when(col("revenue")>500, "Yes").otherwise ("No") )

In [92]:
df_flag.select("trip_id","revenue", "high_value").show()

+------------+-------+----------+
|     trip_id|revenue|high_value|
+------------+-------+----------+
|TRIP00000015|1286.22|       Yes|
|TRIP00000131|1361.45|       Yes|
|TRIP00000153| 377.06|        No|
|TRIP00000171|3262.13|       Yes|
|TRIP00000231| 367.34|        No|
|TRIP00000380| 888.01|       Yes|
|TRIP00000469|4268.24|       Yes|
|TRIP00000520|4296.96|       Yes|
|TRIP00000593|4090.86|       Yes|
|TRIP00000630|4817.64|       Yes|
|TRIP00000751|4070.49|       Yes|
|TRIP00000764|1544.86|       Yes|
|TRIP00000785|1043.07|       Yes|
|TRIP00000844| 2922.2|       Yes|
|TRIP00000855|1241.93|       Yes|
|TRIP00001052| 402.62|        No|
|TRIP00001113|1257.25|       Yes|
|TRIP00001148|3632.52|       Yes|
|TRIP00001240|3783.76|       Yes|
|TRIP00001269|3627.32|       Yes|
+------------+-------+----------+
only showing top 20 rows


In [94]:
df_flag.filter(col("high_value") == "Yes") \
       .select("trip_id", "revenue", "high_value") \
       .show()

+------------+-------+----------+
|     trip_id|revenue|high_value|
+------------+-------+----------+
|TRIP00000015|1286.22|       Yes|
|TRIP00000131|1361.45|       Yes|
|TRIP00000171|3262.13|       Yes|
|TRIP00000380| 888.01|       Yes|
|TRIP00000469|4268.24|       Yes|
|TRIP00000520|4296.96|       Yes|
|TRIP00000593|4090.86|       Yes|
|TRIP00000630|4817.64|       Yes|
|TRIP00000751|4070.49|       Yes|
|TRIP00000764|1544.86|       Yes|
|TRIP00000785|1043.07|       Yes|
|TRIP00000844| 2922.2|       Yes|
|TRIP00000855|1241.93|       Yes|
|TRIP00001113|1257.25|       Yes|
|TRIP00001148|3632.52|       Yes|
|TRIP00001240|3783.76|       Yes|
|TRIP00001269|3627.32|       Yes|
|TRIP00001333|1326.02|       Yes|
|TRIP00001437|1240.24|       Yes|
|TRIP00001525|3799.77|       Yes|
+------------+-------+----------+
only showing top 20 rows


In [95]:
df_pivot = df_master.groupBy("route_id") \
    .pivot("load_type") \
    .sum("revenue")

In [96]:
df_pivot.show()


+--------+------------------+------------------+
|route_id|           Dry Van|      Refrigerated|
+--------+------------------+------------------+
|RTE00006|1062009.2999999998|1043429.6900000008|
|RTE00033| 287693.2599999998| 312497.5899999998|
|RTE00051|2572234.9799999995| 2475059.139999999|
|RTE00011| 685091.4799999997| 697030.6400000014|
|RTE00034|3680712.0399999972|3584259.2700000005|
|RTE00007|2912700.5100000026|3096632.7899999996|
|RTE00056| 3116151.320000003|3117966.9700000007|
|RTE00024| 2238614.770000001|2134481.6699999995|
|RTE00043|1839895.6400000013|1776235.5199999989|
|RTE00016| 5009225.880000004| 5059828.340000003|
|RTE00025|3758248.7899999996|3764758.3400000026|
|RTE00009|1253461.7799999996|1275090.2600000007|
|RTE00022| 764958.0400000006| 777813.8099999989|
|RTE00008|1541014.1600000004|         1374285.7|
|RTE00026|2334370.3299999996|        2572563.34|
|RTE00002|1062095.1100000006|1092196.0100000007|
|RTE00035|1504451.2299999997| 1486197.519999999|
|RTE00049|1407829.41

# Top 3 profitable routes per month

In [97]:
from pyspark.sql.functions import month, row_number
from pyspark.sql.window import Window

df = df_master.withColumn("month", month("load_date"))

agg = df.groupBy("route_id", "month") \
    .sum("revenue")

window_spec = Window.partitionBy("month").orderBy(col("sum(revenue)").desc())

result = agg.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 3)

In [98]:
result.show()

+--------+-----+-----------------+----+
|route_id|month|     sum(revenue)|rank|
+--------+-----+-----------------+----+
|RTE00016|    1|999556.3400000001|   1|
|RTE00029|    1|        865822.46|   2|
|RTE00044|    1|825055.0399999999|   3|
|RTE00044|    2|        815779.23|   1|
|RTE00029|    2|809115.5099999999|   2|
|RTE00016|    2|716930.7099999997|   3|
|RTE00044|    3|875649.2099999997|   1|
|RTE00046|    3|        850009.93|   2|
|RTE00016|    3|813119.7499999998|   3|
|RTE00044|    4|929365.6599999998|   1|
|RTE00012|    4|        862460.52|   2|
|RTE00046|    4|        794451.27|   3|
|RTE00046|    5|913134.1599999999|   1|
|RTE00012|    5|866922.8000000003|   2|
|RTE00016|    5|821873.1999999997|   3|
|RTE00044|    6|947635.7100000002|   1|
|RTE00012|    6|887871.5800000005|   2|
|RTE00016|    6|877092.3399999999|   3|
|RTE00044|    7|867257.5600000002|   1|
|RTE00012|    7|861732.9300000003|   2|
+--------+-----+-----------------+----+
only showing top 20 rows
